# 1. Cost-Sensitive Learning: Handling Class Imbalance with `class_weight`

This notebook covers:
1. **The Imbalance Trap & The Accuracy Paradox**: Why standard models achieve 95%+ accuracy while failing completely on the minority class.
2. **The Mathematics of `class_weight='balanced'`**: How Scikit-Learn scales loss gradients proportionally to class frequencies.
3. **Training Baseline vs. Cost-Sensitive Models**: Comparing unweighted and balanced models across `LogisticRegression` and `RandomForestClassifier`.
4. **Custom Cost Matrices (`class_weight={0: 1, 1: 10}`)**: Tuning penalty ratios to reflect asymmetric business costs.
5. **Proper Evaluation**: Focusing on Confusion Matrix, Recall, Precision, and F1-Score instead of raw Accuracy.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Generate a synthetic fraud detection dataset with a 95:5 class imbalance
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    weights=[0.95, 0.05],  # 95% Class 0 (Legit), 5% Class 1 (Fraud)
    flip_y=0,
    random_state=42
)

# Convert to DataFrame
feature_names = [f"Feature_{i+1}" for i in range(X_raw.shape[1])]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Is_Fraud'] = y_raw

print("=== 1. CLASS DISTRIBUTION ===")
display(df['Is_Fraud'].value_counts(normalize=True).rename('Proportion').to_frame().assign(
    Count=df['Is_Fraud'].value_counts()
))

=== 1. CLASS DISTRIBUTION ===


,Proportion,Count
Is_Fraud,,
0,0.95,950
1,0.05,50


In [3]:
display(df)

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Is_Fraud
0,-2.062806,1.349630,0.592102,-1.679421,3.053408,0.357603,0
1,-1.076196,-0.294657,0.662782,-3.270125,2.622054,0.080713,0
2,-2.407619,1.722726,0.381168,-0.042070,2.502862,1.168427,0
3,-1.279506,1.104792,0.198774,-0.191471,1.546835,1.016459,0
4,0.217687,-3.430056,-0.671144,-0.111399,-1.777715,-2.605821,0
...,...,...,...,...,...,...,...
995,-1.141490,-1.103260,-1.277536,-1.205530,0.957252,-1.025348,0
996,-0.116789,-1.890140,0.881585,0.328725,-1.067478,-1.546102,0
997,-2.121328,-0.596296,0.446894,-2.805718,2.973481,0.888875,0
998,-1.150930,-0.742749,-0.470303,-1.194539,1.092071,-1.331861,0


---
## Part 1: Train / Test Split (Stratified)

We perform an 80/20 train/test split.
* **`stratify=y` is mandatory**: Ensures both the training and test sets contain the exact same 95:5 class distribution.

In [4]:
X = df.drop(columns=['Is_Fraud']).copy()
y = df['Is_Fraud'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=42, stratify=y)

print(f"Train Set Shape: {X_train.shape} | Fraud Cases: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Test Set Shape:  {X_test.shape}  | Fraud Cases: {y_test.sum()} ({y_test.mean():.1%})")

Train Set Shape: (800, 6) | Fraud Cases: 40 (5.0%)
Test Set Shape:  (200, 6)  | Fraud Cases: 10 (5.0%)


---
## Part 2: The Baseline (Unweighted) Model & The Accuracy Paradox

When unweighted, the loss function treats every error equally. Because 95% of rows are Class 0, the model maximizes overall accuracy simply by predicting Class 0 almost every time.

In [5]:
# Train a standard, unweighted Logistic Regression model
baseline_lr = LogisticRegression(random_state=42)
baseline_lr.fit(X_train, y_train)

y_pred_baseline = baseline_lr.predict(X_test)

print("=== BASELINE (UNWEIGHTED) LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_baseline, target_names=['Legit (0)', 'Fraud (1)'], zero_division=0))

=== BASELINE (UNWEIGHTED) LOGISTIC REGRESSION ===
              precision    recall  f1-score   support

   Legit (0)       0.95      1.00      0.97       190
   Fraud (1)       0.00      0.00      0.00        10

    accuracy                           0.95       200
   macro avg       0.47      0.50      0.49       200
weighted avg       0.90      0.95      0.93       200



---
## Part 3: The Math Behind `class_weight='balanced'`

Instead of modifying the dataset (e.g., duplicating or deleting rows), `class_weight='balanced'` **modifies the loss function**.

### The Weight Formula (Scikit-Learn):
$$w_j = \frac{N_{\text{total}}}{K \times N_j}$$

Where:
* $N_{\text{total}}$ = Total number of samples in training data
* $K$ = Total number of classes ($2$ for binary classification)
* $N_j$ = Number of samples in class $j$

#### Concrete Calculation on our Training Set ($N=800$):
* **Class 0 (760 samples):** $w_0 = \frac{800}{2 \times 760} = \frac{800}{1520} \approx \mathbf{0.526}$
* **Class 1 (40 samples):** $w_1 = \frac{800}{2 \times 40} = \frac{800}{80} = \mathbf{10.0}$

**The Result:** The model is penalized **$19\times$ harder** for misclassifying a single Fraud case compared to a Legit case.